In [43]:
#Part 1 – Data Understanding & Preprocessing

!pip install -q pymupdf tiktoken pandas

In [44]:

# Importing libraries and loading the PDF

import pymupdf
import pandas as pd
import re
import os

from google.colab import files

# Upload the ML book PDF
uploaded = files.upload()

# Get PDF filename
pdf_path = next(
    (filename for filename in uploaded.keys()
     if filename.lower().endswith(".pdf")),
    None
)

if pdf_path is None:
    raise ValueError("Please upload a valid PDF file.")

print("PDF loaded successfully:", pdf_path)

# Open PDF
pdf_document = pymupdf.open(pdf_path)

print("Total number of pages:", len(pdf_document))


Saving troubleshooting_runbook.pdf to troubleshooting_runbook (1).pdf
PDF loaded successfully: troubleshooting_runbook (1).pdf
Total number of pages: 7


In [45]:
#Extract text from all pages

# Store extracted content
pages_data = []

for page_number, page in enumerate(pdf_document, start=1):

    # Extract text
    text = page.get_text("text")

    # Basic text cleaning
    text = text.replace("\x00", " ")
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    text = text.strip()

    # Store text and page number
    pages_data.append({
        "page": page_number,
        "text": text
    })

print("Text extraction completed.")

print("Total pages processed:", len(pages_data))

# Display first page's extracted text
print("\nFirst page content:\n")

print(pages_data[0]["text"][:2000])

Text extraction completed.
Total pages processed: 7

First page content:

Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 1
Contoso Platform Troubleshooting Runbook
Document owner: Platform Reliability Team | Version: 3.2 | Last reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consistent way to diagnose and resolve production incidents
across the eight core Contoso services: payment-gateway, auth-service, order-api, inventory-service,
notification-service, search-service, reporting-engine and user-profile-api. It covers the ten most frequent
error codes seen in the incident log over the last two quarters. Each error code has a symptom description,
likely causes, a diagnostic checklist, a fix, and a prevention note.
2. How to use this runbook
Start from the error code in the alert or log line. Confirm the symptom matches, work through the
diagnostic checklist in order, and apply the fix only after the caus

In [46]:
#Explore document structure and text quality

# Create a report for each page

page_statistics = []

for page in pages_data:

    text = page["text"]

    page_statistics.append({
        "Page Number": page["page"],
        "Characters": len(text),
        "Word Count": len(text.split()),
        "Empty Page": len(text.strip()) == 0,
        "Special Characters": text.count("\ufffd")
    })

# Convert to DataFrame
page_df = pd.DataFrame(page_statistics)

# Display page-level report
display(page_df.head(15))

# Overall document statistics
print("\nDOCUMENT SUMMARY")

print("Total Pages:", len(pages_data))

print("Total Characters:",
      page_df["Characters"].sum())

print("Total Words:",
      page_df["Word Count"].sum())

print("Empty Pages:",
      page_df["Empty Page"].sum())

print("Pages with unusual replacement characters:",
      (page_df["Special Characters"] > 0).sum())

,Page Number,Characters,Word Count,Empty Page,Special Characters
0,1,2272,326,False,0
1,2,1890,294,False,0
2,3,2266,354,False,0
3,4,2649,409,False,0
4,5,2695,371,False,0
5,6,2679,380,False,0
6,7,2102,313,False,0



DOCUMENT SUMMARY
Total Pages: 7
Total Characters: 16553
Total Words: 2447
Empty Pages: 0
Pages with unusual replacement characters: 0


In [47]:
# Inspect extracted content

# Search for chapter headings

for page in pages_data:

    text = page["text"]

    if re.search(r'Chapter\s+\d+', text, re.IGNORECASE):

        print(
            f"\nPage {page['page']} contains a chapter heading:"
        )

        print(text[:500])

In [48]:
#Split text into chunks

import tiktoken

# Initialize tokenizer
encoding = tiktoken.get_encoding("cl100k_base")

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100

def create_chunks(text, chunk_size=800, overlap=100):

    tokens = encoding.encode(text)

    chunks = []

    start = 0

    while start < len(tokens):

        end = min(start + chunk_size, len(tokens))

        chunk_tokens = tokens[start:end]

        chunk_text = encoding.decode(chunk_tokens)

        if chunk_text.strip():
            chunks.append(chunk_text.strip())

        if end == len(tokens):
            break

        start = end - overlap

    return chunks


# Generate chunks from all pages

documents = []

for page in pages_data:

    page_chunks = create_chunks(
        page["text"],
        CHUNK_SIZE,
        CHUNK_OVERLAP
    )

    for chunk_number, chunk in enumerate(
        page_chunks, start=1
    ):

        documents.append({
            "text": chunk,
            "page": page["page"],
            "chunk_number": chunk_number
        })


print("Chunking completed.")

print("Total number of chunks:", len(documents))

Chunking completed.
Total number of chunks: 7


In [49]:
#Part 2 – Embedding & Vector Database

#Install and import libraries
!pip install -q sentence-transformers faiss-cpu
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

#Initialize the embedding model

# Load Sentence Transformer model

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [50]:
#Generate embeddings for all chunks

# Extract text from document chunks

chunk_texts = [
    document["text"]
    for document in documents
]

# Generate embeddings

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Convert to float32 for FAISS

embeddings = embeddings.astype("float32")

print("Embedding generation completed.")

print("Embedding shape:", embeddings.shape)

print("Number of embeddings:", len(embeddings))

print("Embedding dimensions:", embeddings.shape[1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding generation completed.
Embedding shape: (7, 384)
Number of embeddings: 7
Embedding dimensions: 384


In [51]:
#Create FAISS vector database

# Get embedding dimensions

dimension = embeddings.shape[1]

# Create FAISS index

vector_database = faiss.IndexFlatIP(dimension)

# Add embeddings to FAISS

vector_database.add(embeddings)

print("FAISS vector database created successfully.")

print("Total vectors stored:",
      vector_database.ntotal)

FAISS vector database created successfully.
Total vectors stored: 7


In [52]:
#Implement similarity search
def similarity_search(query, top_k=5):

    # Convert user query into embedding

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    # Search FAISS database

    scores, indices = vector_database.search(
        query_embedding,
        top_k
    )

    results = []

    # Collect matching chunks

    for score, index in zip(scores[0], indices[0]):

        if index == -1:
            continue

        result = documents[index].copy()

        result["similarity_score"] = float(score)

        results.append(result)

    return results

In [53]:
# Part 3 - Retrieval Pipeline

#Create a reusable retrieval function

def retrieve_documents(query, k=5):

    results = similarity_search(
        query,
        top_k=k
    )

    print("\nUSER QUERY:", query)

    print("\nNumber of retrieved chunks:", len(results))

    for i, result in enumerate(results, start=1):

        print(f"\nRETRIEVED CHUNK {i}")

        print("Page Number:", result["page"])

        print("Similarity Score:",
              round(result["similarity_score"], 4))

        print("Content:")

        print(result["text"][:1000])

        print("-" * 80)

    return results



In [54]:
# Test a sample query

query = "Explain how the k-nearest neighbors algorithm works."

results = retrieve_documents(
    query,
    k=5
)


USER QUERY: Explain how the k-nearest neighbors algorithm works.

Number of retrieved chunks: 5

RETRIEVED CHUNK 1
Page Number: 6
Similarity Score: 0.0636
Content:
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 6
Infrastructure Errors
MEM-OOM - Container OOMKilled
Symptom: Pods restart repeatedly; kubectl describe shows 'Last State: Terminated, Reason: OOMKilled,
Exit Code: 137'. Common on search-service (large result sets) and reporting-engine (big in-memory
aggregations).
Likely causes: Memory limit set below real working set; unbounded caches or result sets; memory leak
introduced by a release; JVM heap configured larger than the container limit; a report requested over an
unusually large date range.
Diagnostics: 1. Check memory graph for a saw-tooth pattern (leak) versus a step (large request). 2.
Confirm the limit: kubectl get pod -o jsonpath='{.spec.containers[*].resources}'. 3. Capture a heap dump
before the next restart. 4. Id

In [55]:
#Experiment with different top-k values

query = "What is the difference between overfitting and underfitting?"

k_values = [3, 5, 7]

for k in k_values:

    print("\n" + "=" * 80)

    print("EXPERIMENT WITH TOP-K =", k)

    print("=" * 80)

    results = similarity_search(
        query,
        top_k=k
    )

    for i, result in enumerate(results, start=1):

        print(f"\nResult {i}")

        print("Page:", result["page"])

        print("Similarity:",
              round(result["similarity_score"], 4))

        print("Content:")

        print(result["text"][:500])


EXPERIMENT WITH TOP-K = 3

Result 1
Page: 2
Similarity: 0.0811
Content:
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 2
Severity and Escalation
5. Severity matrix
Severity
Definition
Ack target
Update
cadence
Escalate to
Sev1
Full outage or data loss risk; revenue
or security impact
5 min
Every 15 min
Engineering Manager + Director
on-call
Sev2
Major feature degraded for many
users; workaround limited
10 min
Every 30 min
Engineering Manager
Sev3
Partial degradation, workaround
available
30 min
Every 2 hours
Owning 

Result 2
Page: 7
Similarity: 0.047
Content:
Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 7
Appendix: Quick Reference and Post-Incident
Checklist
A. Error code quick reference
Code
Meaning
First thing to check
Typical owner
ERR-401
Expired / invalid token
Token TTL config, NTP clock skew, JWKS cache
Team Sentinel
ERR-429
Rate limit exceeded
Top API keys, retry storms
Tea

In [56]:
#Part 4 – Answer Generation (RAG)

#Install Gemini SDK

!pip install -q google-genai

#Configure Gemini API

from google import genai
from google.colab import userdata

from getpass import getpass

GEMINI_API_KEY = getpass("Enter Gemini API Key: ")

client = genai.Client(
    api_key=GEMINI_API_KEY
)

Enter Gemini API Key: ··········


In [57]:

# Gemini model configuration and answer generation

MODEL_NAME = "gemini-3.1-flash-lite"

import time
import random
from google.genai.errors import ServerError

def generate_with_retry(prompt, max_attempts=4):

    for attempt in range(max_attempts):

        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt
            )

            return response.text

        except ServerError as e:

            if attempt == max_attempts - 1:
                raise

            wait_time = (5 * (2 ** attempt)) + random.uniform(0, 2)

            print(
                f"Gemini server is busy. "
                f"Retrying in {wait_time:.1f} seconds..."
            )

            time.sleep(wait_time)


In [62]:

# Generate an answer using retrieved documents and Gemini

def generate_answer(question, top_k=5):

    # Step 1: Retrieve relevant chunks from FAISS
    retrieved_chunks = similarity_search(
        question,
        top_k=top_k
    )

    # Step 2: Prepare context from retrieved chunks
    context = ""

    for i, chunk in enumerate(retrieved_chunks, start=1):

        context += f"""
Source {i}
PDF Page: {chunk['page']}
Content:
{chunk['text']}

"""

    # Step 3: Create a grounded prompt
    prompt = f"""
You are a technical knowledge assistant.

Answer the user's question using ONLY the information
provided in the retrieved context below.

Instructions:
1. Provide a clear and accurate answer.
2. Do not introduce information that is not present in the context.
3. If the context does not contain enough information,
   clearly state that the answer is not available in the document.
4. Do not make assumptions or invent technical details.

Retrieved Context:
{context}

User Question:
{question}

Answer:
"""

    # Step 4: Generate answer using Gemini
    answer = generate_with_retry(prompt)

    # Step 5: Return answer and retrieved source chunks
    return answer, retrieved_chunks


In [63]:
# Demonstrate multiple queries using the revised Troubleshooting Runbook

test_queries = [
    "What are the eight core services covered in the Contoso troubleshooting runbook?",

    "What are the golden rules engineers should follow when handling production incidents?",

    "What are the severity levels Sev1 to Sev4, and what are their acknowledgement and update requirements?",

    "What steps should an on-call engineer follow during the first five minutes of an incident?",

    "What causes ERR-401 authentication token errors, and how can they be diagnosed and resolved?",

    "How should engineers troubleshoot ERR-429 rate limit exceeded errors?",

    "What are the common causes of ERR-500 unhandled exceptions, and when should a deployment rollback be considered?",

    "How should an engineer investigate ERR-503 downstream dependency unavailable errors?",

    "What causes ERR-504 gateway timeouts, and what diagnostic steps should be followed?",

    "How can DB-1205 lock wait timeout errors be identified and resolved in PostgreSQL?",

    "What are the possible causes of database connection pool exhaustion (DB-POOL), and how can it be prevented?",

    "How should engineers diagnose and resolve MEM-OOM container errors in Kubernetes?",

    "What steps should be followed when disk usage exceeds 95 percent and the DISK-FULL error occurs?",

    "How can an expired TLS certificate (CERT-EXP) be diagnosed and renewed?",

    "What information must be recorded during a post-incident review, and which operational metrics does the team track?"
]

for query in test_queries:
    print("\\n" + "=" * 80)
    print("QUESTION:", query)
    response = rag_knowledge_assistant(query)
    print("\\nRESPONSE:\\n")
    print(response)

\n================================================================================
QUESTION: What are the eight core services covered in the Contoso troubleshooting runbook?
\nRESPONSE:\n

# Answer

The eight core Contoso services covered in the troubleshooting runbook are:

1. payment-gateway
2. auth-service
3. order-api
4. inventory-service
5. notification-service
6. search-service
7. reporting-engine
8. user-profile-api

---

# Retrieved Sources



### Source 1

- PDF Page: 1
- Similarity Score: 0.6953

**Retrieved Content:**

Contoso Platform Engineering | Troubleshooting Runbook v3.2 | Internal - Synthetic demo data
Page 1
Contoso Platform Troubleshooting Runbook
Document owner: Platform Reliability Team | Version: 3.2 | Last reviewed: 2026-10-15
1. Purpose and scope
This runbook gives on-call engineers a consistent way to diagnose and resolve production incidents
across the eight core Contoso services: payment-gateway, auth-service, order-api, inventory-service,
notification-serv

In [59]:
# End-to-End RAG Application

!pip install -q gradio

In [65]:
# Create the complete RAG application function

def rag_knowledge_assistant(question):

    if not question or not question.strip():

        return "Please enter a question."

    try:

        # Generate answer using RAG pipeline

        answer, retrieved_chunks = generate_answer(
            question,
            top_k=5
        )

        # Prepare source information

        sources = ""

        for i, chunk in enumerate(
            retrieved_chunks,
            start=1
        ):

            sources += f"""

### Source {i}

- PDF Page: {chunk['page']}
- Similarity Score: {chunk['similarity_score']:.4f}

**Retrieved Content:**

{chunk['text'][:600]}

---

"""

        # Combine answer and sources

        final_response = f"""
# Answer

{answer}

---

# Retrieved Sources

{sources}
"""

        return final_response

    except Exception as e:

        return f"An error occurred: {str(e)}"

In [61]:
# Build the Gradio interface

import gradio as gr

# Create Gradio interface

rag_interface = gr.Interface(

    fn=rag_knowledge_assistant,

    inputs=gr.Textbox(
        lines=3,
        placeholder="Ask any Machine Learning question...",
        label="Enter Your Question"
    ),

    outputs=gr.Markdown(
        label="Knowledge Assistant Response"
    ),

    title="Machine Learning Knowledge Assistant",

    description="""
    An AI-powered Retrieval-Augmented Generation (RAG)
    application built using the Introduction to Machine
    Learning with Python book.

    Ask questions about Machine Learning concepts,
    algorithms, preprocessing and model evaluation.
    """
)

# Launch application

rag_interface.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3b73aef258375daf60.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
